In [ ]:
# Détection de fraude — EDA session 1 (6/08/2026)

Dataset : IEEE-CIS Fraud Detection (Kaggle), `train_transaction.csv`

## Volumétrie
- 590540 lignes × 394 colonnes

## Cible `isFraud`
- Taux de fraude : 3.49 %
- Conséquence : accuracy inutilisable car un modèle qui prédit « légitime » partout atteint 96,5 % d'accuracy en attrapant 0 fraude → métriques : precision / recall

## Types de colonnes
- 380 numériques (float+int) / 14 texte (à encoder plus tard)

## Valeurs manquantes
- 2 colonnes vides à >90 % (pires : colonne "dist2" vide à 93.6 % et colonne "D7" vide à 0.934099 )
- Blocs V à taux de NaN identiques → vides sur les mêmes lignes → même source (blocs livrés par Vesta) → redondance possible à l'intérieur d'un bloc, piste de dégraissage
- Décision : on ne supprime/remplit rien car (1) XGBoost traite nativement le format NaN (2) l'absence peut être corrélée à la fraude elle-même

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/train_transaction.csv")

In [3]:
df.shape

(590540, 394)

In [7]:
df.sample(10)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
153699,3140699,0,3182383,76.95,W,16314,400.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
183400,3170400,0,4057286,107.95,W,11690,111.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
482311,3469311,0,12530425,107.95,W,10680,373.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
246497,3233497,0,5864323,49.00,W,14605,528.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
447832,3434832,1,11401734,105.00,W,17999,482.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
481345,3468345,0,12512639,171.00,W,10616,583.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
379760,3366760,0,9498811,58.95,W,6647,553.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
330365,3317365,0,8117953,67.95,W,7225,583.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
300358,3287358,0,7419937,1104.00,W,17249,543.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
264562,3251562,0,6390363,53.95,W,17131,111.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df["isFraud"].value_counts(normalize=True)

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

In [16]:
df.dtypes.value_counts()
#df.select_dtypes(include="str").columns

float64    376
str         14
int64        4
Name: count, dtype: int64

In [14]:
nan_rates = df.isna().mean()
nan_rates.sort_values(ascending=False).head(20)

dist2    0.936284
D7       0.934099
D13      0.895093
D14      0.894695
D12      0.890410
D6       0.876068
D9       0.873123
D8       0.873123
V153     0.861237
V149     0.861237
V141     0.861237
V146     0.861237
V154     0.861237
V162     0.861237
V142     0.861237
V158     0.861237
V161     0.861237
V157     0.861237
V138     0.861237
V139     0.861237
dtype: float64

In [15]:
(nan_rates>0.9).sum()

np.int64(2)